# **DEFINISI FUNGSI MODULAR**

---



## Setup Fashion MNIST

In [2]:
def setup_fashion_mnist_environment(model_name):
    """
    Menangani mounting drive, setup path, dan inisialisasi DataLoaders.
    """
    from google.colab import drive
    import os
    from torchvision import datasets, transforms
    from torch.utils.data import DataLoader

    # 1. Mount Drive
    drive.mount('/content/drive', force_remount=True)

    # 2. Setup Paths
    model_dir = '/content/drive/MyDrive/model_modular'
    os.makedirs(model_dir, exist_ok=True)
    path = os.path.join(model_dir, model_name)

    # 3. DataLoaders
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])

    train_set = datasets.FashionMNIST(root='./data_fashion', train=True, download=True, transform=transform)
    test_set = datasets.FashionMNIST(root='./data_fashion', train=False, download=True, transform=transform)

    train_loader = DataLoader(train_set, batch_size=128, shuffle=True)
    test_loader = DataLoader(test_set, batch_size=1000, shuffle=False)

    return path, train_loader, test_loader

print("✅ Helper function setup_fashion_mnist_environment siap digunakan.")

✅ Helper function setup_fashion_mnist_environment siap digunakan.


## Load or Training Model & Eval Model

In [3]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm

def evaluate_accuracy(model, data_loader, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in data_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return 100 * correct / total

def load_and_prepare_model(model_class, model_path, train_loader, test_loader, device, num_epochs=10, lr=0.001):
    model = model_class().to(device)
    history = {'loss': [], 'accuracy': []}

    if os.path.exists(model_path):
        print(f"✅ Loading existing model from {model_path}...")
        checkpoint = torch.load(model_path, map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        history = checkpoint.get('history', history)
        print(f"✅ Model loaded. Previous accuracy: {history['accuracy'][-1]:.2f}%" if history['accuracy'] else "✅ Model loaded.")
    else:
        print(f"❌ No model found at {model_path}. Starting training...")
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=lr)

        for epoch in range(num_epochs):
            model.train()
            running_loss = 0.0
            for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
                images, labels = images.to(device), labels.to(device)
                optimizer.zero_grad()
                outputs = model(images)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
                running_loss += loss.item()

            acc = evaluate_accuracy(model, test_loader, device)
            avg_loss = running_loss / len(train_loader)
            history['loss'].append(avg_loss)
            history['accuracy'].append(acc)
            print(f"Epoch {epoch+1}: Loss = {avg_loss:.4f}, Accuracy = {acc:.2f}%")

        print(f"💾 Saving model to {model_path}...")
        torch.save({'model_state_dict': model.state_dict(), 'history': history}, model_path)
        print("✅ Training complete and model saved.")

    return model, history

print("✅ Unified Model Manager & Trainer defined!")

✅ Unified Model Manager & Trainer defined!


## Fungsi Kuantisasi Modular

Standard & Fine-Grained Quantization Ternary


In [4]:
import torch
import torch.nn as nn
import copy
import numpy as np

def standard_ternary_quantize(weight_tensor, asymmetric=False):
    original_shape = weight_tensor.shape
    flat_weights = weight_tensor.flatten()

    if not asymmetric:
        abs_weights = torch.abs(flat_weights)
        if abs_weights.max().item() == 0:
            return torch.zeros_like(weight_tensor), 0.0
        delta = 0.7 * torch.mean(abs_weights)
        mask = abs_weights > delta
        if mask.sum() == 0:
            return torch.zeros_like(weight_tensor), 0.0
        alpha = abs_weights[mask].sum() / mask.sum().float()
        ternary_flat = torch.zeros_like(flat_weights)
        ternary_flat[flat_weights > delta] = alpha
        ternary_flat[flat_weights < -delta] = -alpha
        return ternary_flat.reshape(original_shape), alpha.item()
    else:
        # Asymmetric logic for standard ternary
        pos_mask = flat_weights > 0
        neg_mask = flat_weights < 0

        a_pos, a_neg = 0.0, 0.0
        d_pos, d_neg = 0.0, 0.0

        if pos_mask.any():
            pos_vals = flat_weights[pos_mask]
            d_pos = 0.7 * torch.mean(pos_vals)
            m_pos = pos_vals > d_pos
            if m_pos.any(): a_pos = torch.mean(pos_vals[m_pos])

        if neg_mask.any():
            neg_vals = flat_weights[neg_mask]
            d_neg = 0.7 * torch.mean(torch.abs(neg_vals))
            m_neg = torch.abs(neg_vals) > d_neg
            if m_neg.any(): a_neg = torch.mean(neg_vals[m_neg])

        ternary_flat = torch.zeros_like(flat_weights)
        ternary_flat[flat_weights > d_pos] = a_pos
        ternary_flat[flat_weights < -d_neg] = a_neg

        avg_alpha = (abs(a_pos) + abs(a_neg)) / 2.0
        return ternary_flat.reshape(original_shape), avg_alpha.item() if torch.is_tensor(avg_alpha) else avg_alpha

def apply_standard_ternary_to_model(model, asymmetric=False, verbose=False):
    model_q = copy.deepcopy(model)
    layer_alphas = []
    for name, module in model_q.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            ternary_weight, alpha = standard_ternary_quantize(module.weight.data, asymmetric=asymmetric)
            module.weight.data = ternary_weight
            layer_alphas.append(alpha)
            if verbose: print(f"  ✓ {name}: quantized (Asym={asymmetric}) with α={alpha:.4f}")
    return model_q, layer_alphas

def ternary_quantize_group(weight_tensor, group_size=4, asymmetric=False):
    original_shape = weight_tensor.shape
    flat_weights = weight_tensor.flatten()
    n = flat_weights.numel()
    ternary_flat = torch.zeros_like(flat_weights)
    alphas = []

    for i in range(0, n, group_size):
        group = flat_weights[i:min(i+group_size, n)]
        if len(group) == 0: continue

        if not asymmetric:
            abs_group = torch.abs(group)
            if abs_group.max().item() == 0:
                alphas.append(0.0)
                continue
            thresholds = torch.linspace(abs_group.min().item(), abs_group.max().item(), steps=20)
            best_delta, best_score = 0, -float('inf')
            for delta in thresholds:
                mask = abs_group > delta
                if mask.sum() == 0: continue
                score = (abs_group[mask].sum() ** 2) / mask.sum().float()
                if score > best_score: best_score, best_delta = score, delta

            mask = abs_group > best_delta
            if mask.sum() == 0:
                alphas.append(0.0)
                continue
            alpha = abs_group[mask].sum() / mask.sum().float()
            alphas.append(alpha.item())
            ternary_group = torch.zeros_like(group)
            ternary_group[group > best_delta] = alpha
            ternary_group[group < -best_delta] = -alpha
        else:
            # Asymmetric logic: Separate delta and alpha for positive and negative values
            pos_mask = group > 0
            neg_mask = group < 0

            a_pos, a_neg = 0.0, 0.0
            d_pos, d_neg = 0.0, 0.0

            if pos_mask.any():
                pos_vals = group[pos_mask]
                d_pos = 0.7 * torch.mean(pos_vals)
                m_pos = pos_vals > d_pos
                if m_pos.any(): a_pos = torch.mean(pos_vals[m_pos])

            if neg_mask.any():
                neg_vals = group[neg_mask]
                d_neg = 0.7 * torch.mean(torch.abs(neg_vals))
                m_neg = torch.abs(neg_vals) > d_neg
                if m_neg.any(): a_neg = torch.mean(neg_vals[m_neg])

            ternary_group = torch.zeros_like(group)
            ternary_group[group > d_pos] = a_pos
            ternary_group[group < -d_neg] = a_neg
            alphas.append((abs(a_pos) + abs(a_neg)).item() / 2.0 if torch.is_tensor(a_pos) else (abs(a_pos) + abs(a_neg)) / 2.0)

        ternary_flat[i:min(i+group_size, n)] = ternary_group

    return ternary_flat.reshape(original_shape), alphas

def apply_fgq_to_model(model, group_size=4, asymmetric=False, verbose=False):
    if isinstance(group_size, (list, set, tuple)):
        results = {}
        for gs in sorted(list(group_size)):
            if verbose: print(f"\n--- Processing Group Size: {gs} (Asymmetric={asymmetric}) ---")
            m_q, a_m = apply_fgq_to_model(model, gs, asymmetric, verbose)
            results[gs] = (m_q, a_m)
        return results

    model_q = copy.deepcopy(model)
    all_alpha_means = []
    for name, module in model_q.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            ternary_weight, alphas = ternary_quantize_group(module.weight.data, group_size, asymmetric)
            module.weight.data = ternary_weight
            m_alpha = np.mean(alphas) if alphas else 0
            all_alpha_means.append(m_alpha)
            if verbose: print(f"  ✓ {name}: FGQ N={group_size}, Asym={asymmetric}, mean_́={m_alpha:.4f}")
    return model_q, all_alpha_means

def find_best_fgq_model(fgq_results, test_loader, device):
    best_acc = -1.0
    best_gs = None
    best_model = None

    print("\n--- Evaluating All Group Sizes ---")
    for gs, (model_q, _) in fgq_results.items():
        acc = evaluate_accuracy(model_q, test_loader, device)
        print(f"Group Size {gs}: Accuracy = {acc:.2f}%")
        if acc > best_acc:
            best_acc = acc
            best_gs = gs
            best_model = model_q

    print(f"\n✅ Best Result: Group Size {best_gs} with Accuracy {best_acc:.2f}%")
    return best_gs, best_model, best_acc

## Definisi Model

### LeNet-5

In [5]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# 1. Redefine Model Class
class LeNet5(nn.Module):
    def __init__(self):
        super(LeNet5, self).__init__()
        self.conv1 = nn.Conv2d(1, 6, kernel_size=5, padding=2)
        self.conv2 = nn.Conv2d(6, 16, kernel_size=5)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = nn.functional.avg_pool2d(x, 2)
        x = torch.relu(self.conv2(x))
        x = nn.functional.avg_pool2d(x, 2)
        x = x.view(-1, 16 * 5 * 5)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# 2. Re-initialize DataLoader
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])
test_dataset = datasets.FashionMNIST(root='./data_fashion', train=False, download=True, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)

# 3. Instantiate model and define device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = LeNet5().to(device)

print("✅ Setup for verification complete: Model and Test Loader are ready.")

100%|██████████| 26.4M/26.4M [00:01<00:00, 22.4MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 339kB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 6.31MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 21.4MB/s]

✅ Setup for verification complete: Model and Test Loader are ready.


### ResNet-18

In [6]:
import torch
import torch.nn as nn
from torchvision import models

# 1. Redefine Model Class for ResNet18
class ResNet18(nn.Module):
    def __init__(self, num_classes=10):
        super(ResNet18, self).__init__()
        # Load a pre-trained ResNet18 model
        self.model = models.resnet18(weights=None) # weights=None to avoid downloading ImageNet weights for now

        # Modify the first convolution layer for single-channel input (Fashion MNIST images are grayscale)
        self.model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)

        # Modify the final fully connected layer to match the number of classes (Fashion MNIST has 10 classes)
        num_ftrs = self.model.fc.in_features
        self.model.fc = nn.Linear(num_ftrs, num_classes)

    def forward(self, x):
        return self.model(x)

# Instantiate model and define device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ResNet18().to(device)

print("✅ ResNet18 model defined and ready.")

✅ ResNet18 model defined and ready.


# **PENGUJIAN TRAINING & KUANTISASI**

---



## Model Setup

#### ResNet-18

In [8]:
# 1. Siapkan environment (Drive, Path, Data)
model_path_resnet, train_loader_resnet, test_loader_resnet = setup_fashion_mnist_environment('resnet18_fashion_baseline.pth')

# 2. Panggil manager model
model_fp32_resnet, history_resnet = load_and_prepare_model(
    ResNet18,
    model_path_resnet,
    train_loader_resnet,
    test_loader_resnet,
    device,
    num_epochs=5
)

print(f"\nProses selesai. Akurasi Baseline: {history_resnet['accuracy'][-1]:.2f}%")

Mounted at /content/drive
✅ Loading existing model from /content/drive/MyDrive/model_modular/resnet18_fashion_baseline.pth...
✅ Model loaded. Previous accuracy: 89.91%

Proses selesai. Akurasi Baseline: 89.91%


#### LeNet-5

In [9]:
# 1. Siapkan environment (Drive, Path, Data)
model_path_lenet, train_loader_lenet, test_loader_lenet = setup_fashion_mnist_environment('lenet5_fashion_baseline.pth')

# 2. Panggil manager model
model_fp32_lenet, history_lenet = load_and_prepare_model(
    LeNet5,
    model_path_lenet,
    train_loader_lenet,
    test_loader_lenet,
    device,
    num_epochs=5
)

print(f"\nProses selesai. Akurasi Baseline: {history_lenet['accuracy'][-1]:.2f}%")

Mounted at /content/drive
✅ Loading existing model from /content/drive/MyDrive/model_modular/lenet5_fashion_baseline.pth...
✅ Model loaded. Previous accuracy: 86.17%

Proses selesai. Akurasi Baseline: 86.17%


## Kuantisasi Model

### *Standard Ternary Quantization*

#### ResNet-18

In [9]:
model_ternary_resnet, alphas_resnet = apply_standard_ternary_to_model(model_fp32_resnet, asymmetric=True, verbose=True)
acc_ternary_resnet = evaluate_accuracy(model_ternary_resnet, test_loader, device)

print(f"Baseline: {history_resnet['accuracy'][-1]:.2f}%")
print(f"Ternary: {acc_ternary_resnet:.2f}%")

  ✓ model.conv1: quantized (Asym=True) with α=0.1091
  ✓ model.layer1.0.conv1: quantized (Asym=True) with α=0.0827
  ✓ model.layer1.0.conv2: quantized (Asym=True) with α=0.0821
  ✓ model.layer1.1.conv1: quantized (Asym=True) with α=0.0815
  ✓ model.layer1.1.conv2: quantized (Asym=True) with α=0.0812
  ✓ model.layer2.0.conv1: quantized (Asym=True) with α=0.0620
  ✓ model.layer2.0.conv2: quantized (Asym=True) with α=0.0606
  ✓ model.layer2.0.downsample.0: quantized (Asym=True) with α=0.1540
  ✓ model.layer2.1.conv1: quantized (Asym=True) with α=0.0593
  ✓ model.layer2.1.conv2: quantized (Asym=True) with α=0.0579
  ✓ model.layer3.0.conv1: quantized (Asym=True) with α=0.0431
  ✓ model.layer3.0.conv2: quantized (Asym=True) with α=0.0400
  ✓ model.layer3.0.downsample.0: quantized (Asym=True) with α=0.1116
  ✓ model.layer3.1.conv1: quantized (Asym=True) with α=0.0372
  ✓ model.layer3.1.conv2: quantized (Asym=True) with α=0.0372
  ✓ model.layer4.0.conv1: quantized (Asym=True) with α=0.0284
  ✓

#### LeNet-5

In [10]:
model_ternary_lenet, alphas_lenet = apply_standard_ternary_to_model(model_fp32_lenet, asymmetric=True, verbose=True)
acc_ternary_lenet = evaluate_accuracy(model_ternary_lenet, test_loader, device)

print(f"Baseline: {history_lenet['accuracy'][-1]:.2f}%")
print(f"Ternary: {acc_ternary_lenet:.2f}%")

  ✓ conv1: quantized (Asym=True) with α=0.2316
  ✓ conv2: quantized (Asym=True) with α=0.1075
  ✓ fc1: quantized (Asym=True) with α=0.0722
  ✓ fc2: quantized (Asym=True) with α=0.0841
  ✓ fc3: quantized (Asym=True) with α=0.1215
Baseline: 86.17%
Ternary: 73.48%


### *Fine-Grained Ternary Quantization*

#### ResNet-18

In [11]:
fgq_results_resnet = apply_fgq_to_model(model_fp32_resnet, group_size={4, 8, 16}, asymmetric=True, verbose=True)

best_gs_resnet, best_model_fgq_resnet, best_acc_resnet = find_best_fgq_model(fgq_results_resnet, test_loader_resnet, device)

print(f"\nBaseline Accuracy: {history_resnet['accuracy'][-1]:.2f}%")
print(f"Akurasi Terbaik ditemukan pada Group Size {best_gs_resnet}: {best_acc_resnet:.2f}%")


--- Processing Group Size: 4 (Asymmetric=True) ---
  ✓ model.conv1: FGQ N=4, Asym=True, mean_́=0.0903
  ✓ model.layer1.0.conv1: FGQ N=4, Asym=True, mean_́=0.0664
  ✓ model.layer1.0.conv2: FGQ N=4, Asym=True, mean_́=0.0665
  ✓ model.layer1.1.conv1: FGQ N=4, Asym=True, mean_́=0.0661
  ✓ model.layer1.1.conv2: FGQ N=4, Asym=True, mean_́=0.0660
  ✓ model.layer2.0.conv1: FGQ N=4, Asym=True, mean_́=0.0502
  ✓ model.layer2.0.conv2: FGQ N=4, Asym=True, mean_́=0.0493
  ✓ model.layer2.0.downsample.0: FGQ N=4, Asym=True, mean_́=0.1251
  ✓ model.layer2.1.conv1: FGQ N=4, Asym=True, mean_́=0.0484
  ✓ model.layer2.1.conv2: FGQ N=4, Asym=True, mean_́=0.0470
  ✓ model.layer3.0.conv1: FGQ N=4, Asym=True, mean_́=0.0349
  ✓ model.layer3.0.conv2: FGQ N=4, Asym=True, mean_́=0.0326
  ✓ model.layer3.0.downsample.0: FGQ N=4, Asym=True, mean_́=0.0912
  ✓ model.layer3.1.conv1: FGQ N=4, Asym=True, mean_́=0.0303
  ✓ model.layer3.1.conv2: FGQ N=4, Asym=True, mean_́=0.0304
  ✓ model.layer4.0.conv1: FGQ N=4, Asym=Tru

#### LeNet-5

In [12]:
fgq_results_lenet = apply_fgq_to_model(model_fp32_lenet, group_size={4, 8, 16}, asymmetric=True, verbose=True)

best_gs_lenet, best_model_fgq_lenet, best_acc_lenet = find_best_fgq_model(fgq_results_lenet, test_loader, device)

print(f"\nBaseline Accuracy: {history_lenet['accuracy'][-1]:.2f}%")
print(f"Akurasi Terbaik ditemukan pada Group Size {best_gs_lenet}: {best_acc_lenet:.2f}%")


--- Processing Group Size: 4 (Asymmetric=True) ---
  ✓ conv1: FGQ N=4, Asym=True, mean_́=0.1534
  ✓ conv2: FGQ N=4, Asym=True, mean_́=0.0846
  ✓ fc1: FGQ N=4, Asym=True, mean_́=0.0550
  ✓ fc2: FGQ N=4, Asym=True, mean_́=0.0696
  ✓ fc3: FGQ N=4, Asym=True, mean_́=0.0972

--- Processing Group Size: 8 (Asymmetric=True) ---
  ✓ conv1: FGQ N=8, Asym=True, mean_́=0.1889
  ✓ conv2: FGQ N=8, Asym=True, mean_́=0.0975
  ✓ fc1: FGQ N=8, Asym=True, mean_́=0.0641
  ✓ fc2: FGQ N=8, Asym=True, mean_́=0.0803
  ✓ fc3: FGQ N=8, Asym=True, mean_́=0.1148

--- Processing Group Size: 16 (Asymmetric=True) ---
  ✓ conv1: FGQ N=16, Asym=True, mean_́=0.1915
  ✓ conv2: FGQ N=16, Asym=True, mean_́=0.1030
  ✓ fc1: FGQ N=16, Asym=True, mean_́=0.0676
  ✓ fc2: FGQ N=16, Asym=True, mean_́=0.0833
  ✓ fc3: FGQ N=16, Asym=True, mean_́=0.1169

--- Evaluating All Group Sizes ---
Group Size 4: Accuracy = 82.27%
Group Size 8: Accuracy = 82.33%
Group Size 16: Accuracy = 83.17%

✅ Best Result: Group Size 16 with Accuracy 83.1

## Perbandingan Akurasi

### ResNet18

In [13]:
print(f"Baseline: {history_resnet['accuracy'][-1]:.2f}%")
print(f"Ternary: {acc_ternary_resnet:.2f}%")
print(f"FGQ best Group Size: {best_acc_resnet:.2f}%")

Baseline: 89.91%
Ternary: 24.74%
FGQ best Group Size: 79.02%


### Lenet5

In [14]:
print(f"Baseline: {history_lenet['accuracy'][-1]:.2f}%")
print(f"Ternary: {acc_ternary_lenet:.2f}%")
print(f"FGQ best Group Size: {best_acc_lenet:.2f}%")

Baseline: 86.17%
Ternary: 73.48%
FGQ best Group Size: 83.17%


## Penyimpanan Model Kuantisasi

In [11]:
import torch
import os
import copy
import json
from safetensors.torch import save_file, load_file

def pack_2bit(mask):
    """Pack 4 ternary values into 1 byte (using 0, 1, 2 mapping)"""
    flat = mask.flatten().to(torch.uint8)
    n = flat.numel()
    pad = (4 - (n % 4)) % 4
    if pad > 0:
        flat = torch.cat([flat, torch.zeros(pad, dtype=torch.uint8)])

    # Reshape to (N/4, 4) and pack
    grouped = flat.view(-1, 4)
    packed = (grouped[:, 0] & 3) | ((grouped[:, 1] & 3) << 2) | ((grouped[:, 2] & 3) << 4) | ((grouped[:, 3] & 3) << 6)
    return packed, mask.shape

def unpack_2bit(packed, shape, numel):
    """Unpack uint8 back to ternary mask"""
    unpacked = torch.zeros(packed.numel(), 4, dtype=torch.uint8, device=packed.device)
    unpacked[:, 0] = packed & 3
    unpacked[:, 1] = (packed >> 2) & 3
    unpacked[:, 2] = (packed >> 4) & 3
    unpacked[:, 3] = (packed >> 6) & 3

    mask = unpacked.flatten()[:numel].reshape(shape)
    return mask.to(torch.int8)

def save_ternary_safetensors(model, alphas=None, group_size=None, path='model.safetensors'):
    tensors = {}
    meta_list = []

    for name, module in model.named_modules():
        # --- BatchNorm ---
        if isinstance(module, (nn.BatchNorm2d, nn.BatchNorm1d)):
            tensors[f"{name}.weight"] = module.weight.data.clone()
            tensors[f"{name}.bias"] = module.bias.data.clone()
            tensors[f"{name}.running_mean"] = module.running_mean.clone()
            tensors[f"{name}.running_var"] = module.running_var.clone()
            meta_list.append({"n": name, "type": "bn"})
            continue

        # --- Conv2d / Linear ---
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            w = module.weight.data
            # Buat mask ternary: 0=negatif, 1=no1, 2=positif
            m = torch.zeros_like(w, dtype=torch.int8)
            m[w < 0] = 0
            m[w == 0] = 1
            m[w > 0] = 2

            packed, original_shape = pack_2bit(m)
            tensors[f"{name}.p"] = packed

            info = {
                "n": name,
                "type": "linear",
                "s": list(original_shape),
                "num": w.numel(),
                "has_bias": module.bias is not None
            }

            if module.bias is not None:
                tensors[f"{name}.bias"] = module.bias.data.clone()

            # --- Penanganan Alpha ---
            if alphas is not None:
                # Standard Ternary: satu alpha per layer
                linear_count = len([m for m in meta_list if m.get("type") == "linear"])
                info["a"] = float(alphas[linear_count]) if linear_count < len(alphas) else 1.0
            else:
                # FGQ Asimetris: simpan alpha per group dalam satu tensor (N, 2)
                gs = group_size or 4
                info["gs"] = gs
                flat_w = w.flatten()
                alpha_pairs = []
                for i in range(0, flat_w.numel(), gs):
                    g = flat_w[i:i+gs]
                    pos_vals = g[g > 0]
                    neg_vals = g[g < 0]
                    a_pos = pos_vals.mean().item() if pos_vals.numel() > 0 else 0.0
                    a_neg = neg_vals.mean().item() if neg_vals.numel() > 0 else 0.0
                    alpha_pairs.append([a_pos, a_neg])
                tensors[f"{name}.alpha"] = torch.tensor(alpha_pairs, dtype=torch.float32)  # shape (num_groups, 2)

            meta_list.append(info)

    save_file(tensors, path, metadata={"config": json.dumps(meta_list)})
    print(f"✅ Saved: {path}")

def load_ternary_safetensors(path, template_model):
    from safetensors import safe_open

    tensors = load_file(path)
    with safe_open(path, framework="pt") as f:
        meta_list = json.loads(f.metadata()["config"])

    # Buat salinan model template
    model = copy.deepcopy(template_model)
    state_dict = model.state_dict()

    for info in meta_list:
        name = info["n"]

        # --- BatchNorm ---
        if info.get("type") == "bn":
            state_dict[f"{name}.weight"] = tensors[f"{name}.weight"]
            state_dict[f"{name}.bias"] = tensors[f"{name}.bias"]
            state_dict[f"{name}.running_mean"] = tensors[f"{name}.running_mean"]
            state_dict[f"{name}.running_var"] = tensors[f"{name}.running_var"]
            continue

        # --- Conv2d / Linear ---
        packed = tensors[f"{name}.p"]
        mask = unpack_2bit(packed, info["s"], info["num"])

        # Rekonstruksi bobot dasar (-1, 0, +1)
        w_rec = torch.zeros(info["num"], dtype=torch.float32)
        mask_flat = mask.flatten()
        w_rec[mask_flat == 0] = -1.0
        w_rec[mask_flat == 1] = 0.0
        w_rec[mask_flat == 2] = 1.0

        # --- Terapkan Alpha ---
        if "a" in info:
            # Standard Ternary
            w_rec *= info["a"]
        elif "gs" in info:
            # FGQ Asimetris: alpha per group (tensor (num_groups, 2))
            gs = info["gs"]
            alpha = tensors[f"{name}.alpha"]  # shape (num_groups, 2)
            for i in range(alpha.shape[0]):
                start = i * gs
                end = min(start + gs, info["num"])
                segment = w_rec[start:end]
                # Terapkan alpha positif (kolom 0) ke nilai > 0
                segment[segment > 0] = alpha[i, 0]
                # Terapkan alpha negatif (kolom 1) ke nilai < 0
                segment[segment < 0] = alpha[i, 1]
                w_rec[start:end] = segment

        # Update weight
        state_dict[f"{name}.weight"] = w_rec.reshape(info["s"])

        # Update bias jika ada
        if info.get("has_bias", False) and f"{name}.bias" in tensors:
            state_dict[f"{name}.bias"] = tensors[f"{name}.bias"]

    model.load_state_dict(state_dict)
    return model

### **Eksekusi Penyimpanan Model ke Safetensors**

In [16]:
import os

# 1. Pastikan variabel FGQ tersedia
if 'best_model_fgq_resnet' not in locals():
    print("ⅴ Menghitung ulang FGQ ResNet...")
    fgq_results_resnet = apply_fgq_to_model(model_fp32_resnet, group_size={4}, asymmetric=True)
    best_gs_resnet, best_model_fgq_resnet, best_acc_resnet = find_best_fgq_model(fgq_results_resnet, test_loader_resnet, device)

if 'best_model_fgq_lenet' not in locals():
    print("ⅴ Menghitung ulang FGQ LeNet...")
    fgq_results_lenet = apply_fgq_to_model(model_fp32_lenet, group_size={4}, asymmetric=True)
    best_gs_lenet, best_model_fgq_lenet, best_acc_lenet = find_best_fgq_model(fgq_results_lenet, test_loader_lenet, device)

# 2. Tentukan folder penyimpanan
save_dir = '/content/drive/MyDrive/model_modular/safetensors_models'
os.makedirs(save_dir, exist_ok=True)

# 3. Simpan ResNet-18
save_ternary_safetensors(model_ternary_resnet, alphas=alphas_resnet, path=os.path.join(save_dir, 'resnet18_standard_fashion.safetensors'))
save_ternary_safetensors(best_model_fgq_resnet, group_size=best_gs_resnet, path=os.path.join(save_dir, 'resnet18_fgq_fashion.safetensors'))

# 4. Simpan LeNet-5
save_ternary_safetensors(model_ternary_lenet, alphas=alphas_lenet, path=os.path.join(save_dir, 'lenet5_standard_fashion.safetensors'))
save_ternary_safetensors(best_model_fgq_lenet, group_size=best_gs_lenet, path=os.path.join(save_dir, 'lenet5_fgq_fashion.safetensors'))

print("\n✅ Semua model berhasil disimpan ke Safetensors!")

✅ Saved: /content/drive/MyDrive/model_modular/safetensors_models/resnet18_standard_fashion.safetensors
✅ Saved: /content/drive/MyDrive/model_modular/safetensors_models/resnet18_fgq_fashion.safetensors
✅ Saved: /content/drive/MyDrive/model_modular/safetensors_models/lenet5_standard_fashion.safetensors
✅ Saved: /content/drive/MyDrive/model_modular/safetensors_models/lenet5_fgq_fashion.safetensors

✅ Semua model berhasil disimpan ke Safetensors!


### **Verifikasi Pemuatan (Load) dari Safetensors**

In [17]:
import os

def verify_all_safetensors(save_dir, device):
    model_configs = [
        {'name': 'ResNet-18 Standard fashion', 'file': 'resnet18_standard_fashion.safetensors', 'class': ResNet18, 'loader': test_loader_resnet, 'target': acc_ternary_resnet},
        {'name': 'ResNet-18 FGQ  fashion',      'file': 'resnet18_fgq_fashion.safetensors',      'class': ResNet18, 'loader': test_loader_resnet, 'target': best_acc_resnet},
        {'name': 'LeNet-5 Standard fashion',   'file': 'lenet5_standard_fashion.safetensors',   'class': LeNet5,   'loader': test_loader_lenet,  'target': acc_ternary_lenet},
        {'name': 'LeNet-5 FGQ fashion',        'file': 'lenet5_fgq_fashion.safetensors',        'class': LeNet5,   'loader': test_loader_lenet,  'target': best_acc_lenet},
    ]

    print(f"{'Model Name':<20} | {'Loaded Acc':<12} | {'Target Acc':<12} | {'Status'}")
    print("-" * 65)

    for cfg in model_configs:
        path = os.path.join(save_dir, cfg['file'])
        if not os.path.exists(path):
            print(f"❌ File not found: {cfg['file']}")
            continue

        # Load model
        model = load_ternary_safetensors(path, cfg['class']()).to(device)

        # Evaluasi
        acc = evaluate_accuracy(model, cfg['loader'], device)

        status = "✅ MATCH" if abs(acc - cfg['target']) < 0.01 else "⚠️ MISMATCH"
        print(f"{cfg['name']:<20} | {acc:>10.2f}% | {cfg['target']:>10.2f}% | {status}")

# Eksekusi Verifikasi
verify_all_safetensors(save_dir, device)

Model Name           | Loaded Acc   | Target Acc   | Status
-----------------------------------------------------------------
ResNet-18 Standard fashion |      24.55% |      24.74% | ⚠️ MISMATCH
ResNet-18 FGQ  fashion |      79.02% |      79.02% | ✅ MATCH
LeNet-5 Standard fashion |      73.73% |      73.48% | ⚠️ MISMATCH
LeNet-5 FGQ fashion  |      83.17% |      83.17% | ✅ MATCH


# **INFERENSI**

---



#### ResNet-18 Standard

In [48]:
import torch
import time
import numpy as np
from PIL import Image
from google.colab import files
import os

# ==========================================
# 1. KONFIGURASI FASHION-MNIST
# ==========================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🔥 Menggunakan device: {device}")

# Parameter normalisasi standar Fashion-MNIST
FMNIST_MEAN = 0.2860
FMNIST_STD = 0.3530

# Mapping label Fashion-MNIST ke nama kelas
FASHION_LABELS = [
    'T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
    'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot'
]

# Sesuaikan path dengan lokasi model Fashion-MNIST Anda
base_dir = '/content/drive/MyDrive/model_modular'  # ganti jika perlu
baseline_path = os.path.join(base_dir, 'resnet18_fashion_baseline.pth')  # contoh nama
quantized_path = os.path.join(base_dir, 'safetensors_models', 'resnet18_standard_fashion.safetensors')

# ==========================================
# 4. LOAD MODEL BASELINE (FP32) - FASHION
# ==========================================
print("\n📦 Memuat model Baseline Fashion-MNIST (FP32)...")
model_baseline = ResNet18().to(device)  # arsitektur sama
checkpoint = torch.load(baseline_path, map_location=device)
model_baseline.load_state_dict(checkpoint['model_state_dict'])
model_baseline.eval()
print("✅ Baseline siap.")

# ==========================================
# 5. LOAD MODEL KUANTISASI (FASHION)
# ==========================================
print("\n📦 Memuat model Kuantisasi Fashion-MNIST (Standard Ternary)...")
model_quant = load_ternary_safetensors(quantized_path, ResNet18()).to(device)
model_quant.eval()
print("✅ Model kuantisasi siap.")

# ==========================================
# 6. FUNGSI BENCHMARK KECEPATAN
# ==========================================
def benchmark_speed(model, input_tensor, num_runs=100, warmup_runs=20):
    """Mengukur kecepatan inferensi rata-rata (ms dan FPS)."""
    # Warmup
    for _ in range(warmup_runs):
        with torch.no_grad():
            _ = model(input_tensor)

    if device.type == 'cuda':
        torch.cuda.synchronize()

    start_time = time.time()
    for _ in range(num_runs):
        with torch.no_grad():
            _ = model(input_tensor)

    if device.type == 'cuda':
        torch.cuda.synchronize()
    end_time = time.time()

    total_time_s = end_time - start_time
    avg_time_ms = (total_time_s / num_runs) * 1000
    fps = 1000 / avg_time_ms
    return avg_time_ms, fps

🔥 Menggunakan device: cpu

📦 Memuat model Baseline Fashion-MNIST (FP32)...
✅ Baseline siap.

📦 Memuat model Kuantisasi Fashion-MNIST (Standard Ternary)...
✅ Model kuantisasi siap.


In [51]:
# ==========================================
# 2. UPLOAD GAMBAR CUSTOM
# ==========================================
print("📤 Silakan upload gambar fashion (sepatu, baju, tas, dll) format JPG/PNG:")
uploaded = files.upload()
image_path = list(uploaded.keys())[0]
print(f"✅ Berhasil upload: {image_path}")

# ==========================================
# 3. FUNGSI PREPROSES (FASHION-MNIST STANDARD)
# ==========================================
def preprocess_fashion_image(path, mean=FMNIST_MEAN, std=FMNIST_STD):
    """
    Mengubah gambar custom menjadi format Fashion-MNIST (28x28, grayscale).
    """
    # Buka dan konversi ke grayscale
    img = Image.open(path).convert('L')
    # Resize ke 28x28 (standar Fashion-MNIST)
    img = img.resize((28, 28), Image.Resampling.LANCZOS)

    # Konversi ke numpy array dan normalisasi ke [0, 1]
    img_array = np.array(img, dtype=np.float32) / 255.0

    # Normalisasi standar Fashion-MNIST
    img_tensor = (img_array - mean) / std

    # Tambahkan dimensi batch dan channel -> (1, 1, 28, 28)
    img_tensor = torch.from_numpy(img_tensor).unsqueeze(0).unsqueeze(0)
    return img_tensor.to(device)

# Proses gambar
input_tensor = preprocess_fashion_image(image_path)
print(f"✅ Shape input tensor: {input_tensor.shape}")

# ==========================================
# 7. EKSEKUSI BENCHMARK & PREDIKSI
# ==========================================
print("\n" + "="*60)
print("🚀 Benchmarking Kecepatan Inferensi - Fashion-MNIST")
print("="*60)

# --- Baseline ---
print("\n🔹 Baseline (FP32):")
avg_t, fps = benchmark_speed(model_baseline, input_tensor)
with torch.no_grad():
    pred_idx = model_baseline(input_tensor).argmax().item()
    pred_label = FASHION_LABELS[pred_idx]
print(f"   ⏱️  Rata-rata waktu: {avg_t:.4f} ms")
print(f"   📈 FPS: {fps:.2f} gambar/detik")
print(f"   🎯 Prediksi: {pred_label} (index {pred_idx})")

# --- Kuantisasi ---
print("\n🔹 Kuantisasi Standard Ternary:")
avg_t_q, fps_q = benchmark_speed(model_quant, input_tensor)
with torch.no_grad():
    pred_idx_q = model_quant(input_tensor).argmax().item()
    pred_label_q = FASHION_LABELS[pred_idx_q]
print(f"   ⏱️  Rata-rata waktu: {avg_t_q:.4f} ms")
print(f"   📈 FPS: {fps_q:.2f} gambar/detik")
print(f"   🎯 Prediksi: {pred_label_q} (index {pred_idx_q})")

# ==========================================
# 8. PERBANDINGAN SPEEDUP & VERIFIKASI
# ==========================================
speedup = avg_t / avg_t_q
print("\n" + "="*60)
print("📊 RINGKASAN PERBANDINGAN")
print("="*60)
print(f"⚡ Speedup (Baseline vs Kuantisasi): {speedup:.2f}x lebih cepat")
print(f"✅ Hasil prediksi sama? {'Ya ✅' if pred_idx == pred_idx_q else 'Tidak ❌'}")

# Tampilkan gambar asli
try:
    from IPython.display import display
    print("\n🖼️ Gambar yang diupload:")
    display(Image.open(image_path))
except:
    pass

📤 Silakan upload gambar fashion (sepatu, baju, tas, dll) format JPG/PNG:


Saving 938.png to 938 (2).png
✅ Berhasil upload: 938 (2).png
✅ Shape input tensor: torch.Size([1, 1, 28, 28])

🚀 Benchmarking Kecepatan Inferensi - Fashion-MNIST

🔹 Baseline (FP32):
   ⏱️  Rata-rata waktu: 11.9515 ms
   📈 FPS: 83.67 gambar/detik
   🎯 Prediksi: Shirt (index 6)

🔹 Kuantisasi Standard Ternary:
   ⏱️  Rata-rata waktu: 11.9068 ms
   📈 FPS: 83.99 gambar/detik
   🎯 Prediksi: Pullover (index 2)

📊 RINGKASAN PERBANDINGAN
⚡ Speedup (Baseline vs Kuantisasi): 1.00x lebih cepat
✅ Hasil prediksi sama? Tidak ❌

🖼️ Gambar yang diupload:


#### ResNet-18 Fine-Grained

In [15]:
import torch
import time
import numpy as np
from PIL import Image
from google.colab import files
import os

# ==========================================
# 1. KONFIGURASI FASHION-MNIST
# ==========================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🔥 Menggunakan device: {device}")

# Parameter normalisasi standar Fashion-MNIST
FMNIST_MEAN = 0.2860
FMNIST_STD = 0.3530

# Mapping label Fashion-MNIST ke nama kelas
FASHION_LABELS = [
    'T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
    'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot'
]

# Sesuaikan path dengan lokasi model Fashion-MNIST Anda
base_dir = '/content/drive/MyDrive/model_modular'  # ganti jika perlu
baseline_path_fgq = os.path.join(base_dir, 'resnet18_fashion_baseline.pth')  # contoh nama
quantized_path_fgq = os.path.join(base_dir, 'safetensors_models', 'resnet18_fgq_fashion.safetensors')

# ==========================================
# 4. LOAD MODEL BASELINE (FP32) - FASHION
# ==========================================
print("\n📦 Memuat model Baseline Fashion-MNIST (FP32)...")
model_baseline = ResNet18().to(device)  # arsitektur sama
checkpoint = torch.load(baseline_path, map_location=device)
model_baseline.load_state_dict(checkpoint['model_state_dict'])
model_baseline.eval()
print("✅ Baseline siap.")

# ==========================================
# 5. LOAD MODEL KUANTISASI (FASHION)
# ==========================================
print("\n📦 Memuat model Kuantisasi Fashion-MNIST (Fine-Grained Ternary)...")
model_quant_fgq = load_ternary_safetensors(quantized_path, ResNet18()).to(device)
model_quant_fgq.eval()
print("✅ Model kuantisasi siap.")

# ==========================================
# 6. FUNGSI BENCHMARK KECEPATAN
# ==========================================
def benchmark_speed(model, input_tensor, num_runs=100, warmup_runs=20):
    """Mengukur kecepatan inferensi rata-rata (ms dan FPS)."""
    # Warmup
    for _ in range(warmup_runs):
        with torch.no_grad():
            _ = model(input_tensor)

    if device.type == 'cuda':
        torch.cuda.synchronize()

    start_time = time.time()
    for _ in range(num_runs):
        with torch.no_grad():
            _ = model(input_tensor)

    if device.type == 'cuda':
        torch.cuda.synchronize()
    end_time = time.time()

    total_time_s = end_time - start_time
    avg_time_ms = (total_time_s / num_runs) * 1000
    fps = 1000 / avg_time_ms
    return avg_time_ms, fps

🔥 Menggunakan device: cpu

📦 Memuat model Baseline Fashion-MNIST (FP32)...
✅ Baseline siap.

📦 Memuat model Kuantisasi Fashion-MNIST (Fine-Grained Ternary)...
✅ Model kuantisasi siap.


In [17]:
# ==========================================
# 2. UPLOAD GAMBAR CUSTOM
# ==========================================
print("📤 Silakan upload gambar fashion (sepatu, baju, tas, dll) format JPG/PNG:")
uploaded = files.upload()
image_path = list(uploaded.keys())[0]
print(f"✅ Berhasil upload: {image_path}")

# ==========================================
# 3. FUNGSI PREPROSES (FASHION-MNIST STANDARD)
# ==========================================
def preprocess_fashion_image(path, mean=FMNIST_MEAN, std=FMNIST_STD):
    """
    Mengubah gambar custom menjadi format Fashion-MNIST (28x28, grayscale).
    """
    # Buka dan konversi ke grayscale
    img = Image.open(path).convert('L')
    # Resize ke 28x28 (standar Fashion-MNIST)
    img = img.resize((28, 28), Image.Resampling.LANCZOS)

    # Konversi ke numpy array dan normalisasi ke [0, 1]
    img_array = np.array(img, dtype=np.float32) / 255.0

    # Normalisasi standar Fashion-MNIST
    img_tensor = (img_array - mean) / std

    # Tambahkan dimensi batch dan channel -> (1, 1, 28, 28)
    img_tensor = torch.from_numpy(img_tensor).unsqueeze(0).unsqueeze(0)
    return img_tensor.to(device)

# Proses gambar
input_tensor = preprocess_fashion_image(image_path)
print(f"✅ Shape input tensor: {input_tensor.shape}")

# ==========================================
# 7. EKSEKUSI BENCHMARK & PREDIKSI
# ==========================================
print("\n" + "="*60)
print("🚀 Benchmarking Kecepatan Inferensi - Fashion-MNIST")
print("="*60)

# --- Baseline ---
print("\n🔹 Baseline (FP32):")
avg_t, fps = benchmark_speed(model_baseline, input_tensor)
with torch.no_grad():
    pred_idx = model_baseline(input_tensor).argmax().item()
    pred_label = FASHION_LABELS[pred_idx]
print(f"   ⏱️  Rata-rata waktu: {avg_t:.4f} ms")
print(f"   📈 FPS: {fps:.2f} gambar/detik")
print(f"   🎯 Prediksi: {pred_label} (index {pred_idx})")

# --- Kuantisasi ---
print("\n🔹 Kuantisasi Fine-Grained Ternary:")
avg_t_q, fps_q = benchmark_speed(model_quant_fgq, input_tensor)
with torch.no_grad():
    pred_idx_q = model_quant_fgq(input_tensor).argmax().item()
    pred_label_q = FASHION_LABELS[pred_idx_q]
print(f"   ⏱️  Rata-rata waktu: {avg_t_q:.4f} ms")
print(f"   📈 FPS: {fps_q:.2f} gambar/detik")
print(f"   🎯 Prediksi: {pred_label_q} (index {pred_idx_q})")

# ==========================================
# 8. PERBANDINGAN SPEEDUP & VERIFIKASI
# ==========================================
speedup = avg_t / avg_t_q
print("\n" + "="*60)
print("📊 RINGKASAN PERBANDINGAN")
print("="*60)
print(f"⚡ Speedup (Baseline vs Kuantisasi): {speedup:.2f}x lebih cepat")
print(f"✅ Hasil prediksi sama? {'Ya ✅' if pred_idx == pred_idx_q else 'Tidak ❌'}")

# Tampilkan gambar asli
try:
    from IPython.display import display
    print("\n🖼️ Gambar yang diupload:")
    display(Image.open(image_path))
except:
    pass

📤 Silakan upload gambar fashion (sepatu, baju, tas, dll) format JPG/PNG:


Saving 932.png to 932.png
✅ Berhasil upload: 932.png
✅ Shape input tensor: torch.Size([1, 1, 28, 28])

🚀 Benchmarking Kecepatan Inferensi - Fashion-MNIST

🔹 Baseline (FP32):
   ⏱️  Rata-rata waktu: 9.8339 ms
   📈 FPS: 101.69 gambar/detik
   🎯 Prediksi: Coat (index 4)

🔹 Kuantisasi Fine-Grained Ternary:
   ⏱️  Rata-rata waktu: 12.2954 ms
   📈 FPS: 81.33 gambar/detik
   🎯 Prediksi: Coat (index 4)

📊 RINGKASAN PERBANDINGAN
⚡ Speedup (Baseline vs Kuantisasi): 0.80x lebih cepat
✅ Hasil prediksi sama? Ya ✅

🖼️ Gambar yang diupload:


#### LeNet-5 Standard

In [53]:
import torch
import time
import numpy as np
from PIL import Image
from google.colab import files
import os

# ==========================================
# 1. KONFIGURASI FASHION-MNIST
# ==========================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🔥 Menggunakan device: {device}")

# Parameter normalisasi standar Fashion-MNIST
FMNIST_MEAN = 0.2860
FMNIST_STD = 0.3530

# Mapping label Fashion-MNIST ke nama kelas
FASHION_LABELS = [
    'T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
    'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot'
]

# Sesuaikan path dengan lokasi model Fashion-MNIST Anda
base_dir = '/content/drive/MyDrive/model_modular'  # ganti jika perlu
baseline_path = os.path.join(base_dir, 'lenet5_fashion_baseline.pth')  # contoh nama
quantized_path = os.path.join(base_dir, 'safetensors_models', 'lenet5_standard_fashion.safetensors')

# ==========================================
# 4. LOAD MODEL BASELINE (FP32) - FASHION
# ==========================================
print("\n📦 Memuat model Baseline Fashion-MNIST (FP32)...")
model_baseline = LeNet5().to(device)  # arsitektur sama
checkpoint = torch.load(baseline_path, map_location=device)
model_baseline.load_state_dict(checkpoint['model_state_dict'])
model_baseline.eval()
print("✅ Baseline siap.")

# ==========================================
# 5. LOAD MODEL KUANTISASI (FASHION)
# ==========================================
print("\n📦 Memuat model Kuantisasi Fashion-MNIST (Standard Ternary)...")
model_quant = load_ternary_safetensors(quantized_path, LeNet5()).to(device)
model_quant.eval()
print("✅ Model kuantisasi siap.")

# ==========================================
# 6. FUNGSI BENCHMARK KECEPATAN
# ==========================================
def benchmark_speed(model, input_tensor, num_runs=100, warmup_runs=20):
    """Mengukur kecepatan inferensi rata-rata (ms dan FPS)."""
    # Warmup
    for _ in range(warmup_runs):
        with torch.no_grad():
            _ = model(input_tensor)

    if device.type == 'cuda':
        torch.cuda.synchronize()

    start_time = time.time()
    for _ in range(num_runs):
        with torch.no_grad():
            _ = model(input_tensor)

    if device.type == 'cuda':
        torch.cuda.synchronize()
    end_time = time.time()

    total_time_s = end_time - start_time
    avg_time_ms = (total_time_s / num_runs) * 1000
    fps = 1000 / avg_time_ms
    return avg_time_ms, fps

🔥 Menggunakan device: cpu

📦 Memuat model Baseline Fashion-MNIST (FP32)...
✅ Baseline siap.

📦 Memuat model Kuantisasi Fashion-MNIST (Standard Ternary)...
✅ Model kuantisasi siap.


In [54]:
# ==========================================
# 2. UPLOAD GAMBAR CUSTOM
# ==========================================
print("📤 Silakan upload gambar fashion (sepatu, baju, tas, dll) format JPG/PNG:")
uploaded = files.upload()
image_path = list(uploaded.keys())[0]
print(f"✅ Berhasil upload: {image_path}")

# ==========================================
# 3. FUNGSI PREPROSES (FASHION-MNIST STANDARD)
# ==========================================
def preprocess_fashion_image(path, mean=FMNIST_MEAN, std=FMNIST_STD):
    """
    Mengubah gambar custom menjadi format Fashion-MNIST (28x28, grayscale).
    """
    # Buka dan konversi ke grayscale
    img = Image.open(path).convert('L')
    # Resize ke 28x28 (standar Fashion-MNIST)
    img = img.resize((28, 28), Image.Resampling.LANCZOS)

    # Konversi ke numpy array dan normalisasi ke [0, 1]
    img_array = np.array(img, dtype=np.float32) / 255.0

    # Normalisasi standar Fashion-MNIST
    img_tensor = (img_array - mean) / std

    # Tambahkan dimensi batch dan channel -> (1, 1, 28, 28)
    img_tensor = torch.from_numpy(img_tensor).unsqueeze(0).unsqueeze(0)
    return img_tensor.to(device)

# Proses gambar
input_tensor = preprocess_fashion_image(image_path)
print(f"✅ Shape input tensor: {input_tensor.shape}")


# ==========================================
# 7. EKSEKUSI BENCHMARK & PREDIKSI
# ==========================================
print("\n" + "="*60)
print("🚀 Benchmarking Kecepatan Inferensi - Fashion-MNIST")
print("="*60)

# --- Baseline ---
print("\n🔹 Baseline (FP32):")
avg_t, fps = benchmark_speed(model_baseline, input_tensor)
with torch.no_grad():
    pred_idx = model_baseline(input_tensor).argmax().item()
    pred_label = FASHION_LABELS[pred_idx]
print(f"   ⏱️  Rata-rata waktu: {avg_t:.4f} ms")
print(f"   📈 FPS: {fps:.2f} gambar/detik")
print(f"   🎯 Prediksi: {pred_label} (index {pred_idx})")

# --- Kuantisasi ---
print("\n🔹 Kuantisasi Standard Ternary:")
avg_t_q, fps_q = benchmark_speed(model_quant, input_tensor)
with torch.no_grad():
    pred_idx_q = model_quant(input_tensor).argmax().item()
    pred_label_q = FASHION_LABELS[pred_idx_q]
print(f"   ⏱️  Rata-rata waktu: {avg_t_q:.4f} ms")
print(f"   📈 FPS: {fps_q:.2f} gambar/detik")
print(f"   🎯 Prediksi: {pred_label_q} (index {pred_idx_q})")

# ==========================================
# 8. PERBANDINGAN SPEEDUP & VERIFIKASI
# ==========================================
speedup = avg_t / avg_t_q
print("\n" + "="*60)
print("📊 RINGKASAN PERBANDINGAN")
print("="*60)
print(f"⚡ Speedup (Baseline vs Kuantisasi): {speedup:.2f}x lebih cepat")
print(f"✅ Hasil prediksi sama? {'Ya ✅' if pred_idx == pred_idx_q else 'Tidak ❌'}")

# Tampilkan gambar asli
try:
    from IPython.display import display
    print("\n🖼️ Gambar yang diupload:")
    display(Image.open(image_path))
except:
    pass

📤 Silakan upload gambar fashion (sepatu, baju, tas, dll) format JPG/PNG:


Saving 938.png to 938 (3).png
✅ Berhasil upload: 938 (3).png
✅ Shape input tensor: torch.Size([1, 1, 28, 28])

🚀 Benchmarking Kecepatan Inferensi - Fashion-MNIST

🔹 Baseline (FP32):
   ⏱️  Rata-rata waktu: 0.4256 ms
   📈 FPS: 2349.86 gambar/detik
   🎯 Prediksi: Shirt (index 6)

🔹 Kuantisasi Standard Ternary:
   ⏱️  Rata-rata waktu: 0.4348 ms
   📈 FPS: 2299.95 gambar/detik
   🎯 Prediksi: Pullover (index 2)

📊 RINGKASAN PERBANDINGAN
⚡ Speedup (Baseline vs Kuantisasi): 0.98x lebih cepat
✅ Hasil prediksi sama? Tidak ❌

🖼️ Gambar yang diupload:


#### LeNet-5 Fine-Grained

In [56]:
import torch
import time
import numpy as np
from PIL import Image
from google.colab import files
import os

# ==========================================
# 1. KONFIGURASI FASHION-MNIST
# ==========================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🔥 Menggunakan device: {device}")

# Parameter normalisasi standar Fashion-MNIST
FMNIST_MEAN = 0.2860
FMNIST_STD = 0.3530

# Mapping label Fashion-MNIST ke nama kelas
FASHION_LABELS = [
    'T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
    'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot'
]

# Sesuaikan path dengan lokasi model Fashion-MNIST Anda
base_dir = '/content/drive/MyDrive/model_modular'  # ganti jika perlu
baseline_path = os.path.join(base_dir, 'lenet5_fashion_baseline.pth')  # contoh nama
quantized_path_fgq = os.path.join(base_dir, 'safetensors_models', 'lenet5_fgq_fashion.safetensors')

# ==========================================
# 4. LOAD MODEL BASELINE (FP32) - FASHION
# ==========================================
print("\n📦 Memuat model Baseline Fashion-MNIST (FP32)...")
model_baseline = LeNet5().to(device)  # arsitektur sama
checkpoint = torch.load(baseline_path, map_location=device)
model_baseline.load_state_dict(checkpoint['model_state_dict'])
model_baseline.eval()
print("✅ Baseline siap.")

# ==========================================
# 5. LOAD MODEL KUANTISASI (FASHION)
# ==========================================
print("\n📦 Memuat model Kuantisasi Fashion-MNIST (Fine-Grained Ternary)...")
model_quant_fgq = load_ternary_safetensors(quantized_path_fgq, LeNet5()).to(device)
model_quant_fgq.eval()
print("✅ Model kuantisasi siap.")

# ==========================================
# 6. FUNGSI BENCHMARK KECEPATAN
# ==========================================
def benchmark_speed(model, input_tensor, num_runs=100, warmup_runs=20):
    """Mengukur kecepatan inferensi rata-rata (ms dan FPS)."""
    # Warmup
    for _ in range(warmup_runs):
        with torch.no_grad():
            _ = model(input_tensor)

    if device.type == 'cuda':
        torch.cuda.synchronize()

    start_time = time.time()
    for _ in range(num_runs):
        with torch.no_grad():
            _ = model(input_tensor)

    if device.type == 'cuda':
        torch.cuda.synchronize()
    end_time = time.time()

    total_time_s = end_time - start_time
    avg_time_ms = (total_time_s / num_runs) * 1000
    fps = 1000 / avg_time_ms
    return avg_time_ms, fps

🔥 Menggunakan device: cpu

📦 Memuat model Baseline Fashion-MNIST (FP32)...
✅ Baseline siap.

📦 Memuat model Kuantisasi Fashion-MNIST (Fine-Grained Ternary)...
✅ Model kuantisasi siap.


In [57]:
# ==========================================
# 2. UPLOAD GAMBAR CUSTOM
# ==========================================
print("📤 Silakan upload gambar fashion (sepatu, baju, tas, dll) format JPG/PNG:")
uploaded = files.upload()
image_path = list(uploaded.keys())[0]
print(f"✅ Berhasil upload: {image_path}")

# ==========================================
# 3. FUNGSI PREPROSES (FASHION-MNIST FINE-GRAINED)
# ==========================================
def preprocess_fashion_image(path, mean=FMNIST_MEAN, std=FMNIST_STD):
    """
    Mengubah gambar custom menjadi format Fashion-MNIST (28x28, grayscale).
    """
    # Buka dan konversi ke grayscale
    img = Image.open(path).convert('L')
    # Resize ke 28x28 (fine-grained Fashion-MNIST)
    img = img.resize((28, 28), Image.Resampling.LANCZOS)

    # Konversi ke numpy array dan normalisasi ke [0, 1]
    img_array = np.array(img, dtype=np.float32) / 255.0

    # Normalisasi fine-grained Fashion-MNIST
    img_tensor = (img_array - mean) / std

    # Tambahkan dimensi batch dan channel -> (1, 1, 28, 28)
    img_tensor = torch.from_numpy(img_tensor).unsqueeze(0).unsqueeze(0)
    return img_tensor.to(device)

# Proses gambar
input_tensor = preprocess_fashion_image(image_path)
print(f"✅ Shape input tensor: {input_tensor.shape}")


# ==========================================
# 7. EKSEKUSI BENCHMARK & PREDIKSI
# ==========================================
print("\n" + "="*60)
print("🚀 Benchmarking Kecepatan Inferensi - Fashion-MNIST")
print("="*60)

# --- Baseline ---
print("\n🔹 Baseline (FP32):")
avg_t, fps = benchmark_speed(model_baseline, input_tensor)
with torch.no_grad():
    pred_idx = model_baseline(input_tensor).argmax().item()
    pred_label = FASHION_LABELS[pred_idx]
print(f"   ⏱️  Rata-rata waktu: {avg_t:.4f} ms")
print(f"   📈 FPS: {fps:.2f} gambar/detik")
print(f"   🎯 Prediksi: {pred_label} (index {pred_idx})")

# --- Kuantisasi ---
print("\n🔹 Kuantisasi Fine-Grained Ternary:")
avg_t_q, fps_q = benchmark_speed(model_quant_fgq, input_tensor)
with torch.no_grad():
    pred_idx_q = model_quant_fgq(input_tensor).argmax().item()
    pred_label_q = FASHION_LABELS[pred_idx_q]
print(f"   ⏱️  Rata-rata waktu: {avg_t_q:.4f} ms")
print(f"   📈 FPS: {fps_q:.2f} gambar/detik")
print(f"   🎯 Prediksi: {pred_label_q} (index {pred_idx_q})")

# ==========================================
# 8. PERBANDINGAN SPEEDUP & VERIFIKASI
# ==========================================
speedup = avg_t / avg_t_q
print("\n" + "="*60)
print("📊 RINGKASAN PERBANDINGAN")
print("="*60)
print(f"⚡ Speedup (Baseline vs Kuantisasi): {speedup:.2f}x lebih cepat")
print(f"✅ Hasil prediksi sama? {'Ya ✅' if pred_idx == pred_idx_q else 'Tidak ❌'}")

# Tampilkan gambar asli
try:
    from IPython.display import display
    print("\n🖼️ Gambar yang diupload:")
    display(Image.open(image_path))
except:
    pass

📤 Silakan upload gambar fashion (sepatu, baju, tas, dll) format JPG/PNG:


Saving 938.png to 938 (4).png
✅ Berhasil upload: 938 (4).png
✅ Shape input tensor: torch.Size([1, 1, 28, 28])

🚀 Benchmarking Kecepatan Inferensi - Fashion-MNIST

🔹 Baseline (FP32):
   ⏱️  Rata-rata waktu: 0.5768 ms
   📈 FPS: 1733.76 gambar/detik
   🎯 Prediksi: Shirt (index 6)

🔹 Kuantisasi Fine-Grained Ternary:
   ⏱️  Rata-rata waktu: 0.4029 ms
   📈 FPS: 2481.91 gambar/detik
   🎯 Prediksi: Shirt (index 6)

📊 RINGKASAN PERBANDINGAN
⚡ Speedup (Baseline vs Kuantisasi): 1.43x lebih cepat
✅ Hasil prediksi sama? Ya ✅

🖼️ Gambar yang diupload:
